## **Create a table in silver_supplier Catalog which contains the continent of each supplier location. Generative AI is used to create that column** ##

In [1]:
bronze_catalog = "bronze_supplier"
silver_catalog = "silver_supplier"
bronze_schema = "supplier_schema"
silver_schema = "supplier_schema"
bronze_table_extended_csv = "bronze_supplier_basic_Cont_csv"
silver_supplier_basic_Cont_csv = "silver_supplier_basic_Cont_csv"
silver_supplier_basic_AI_csv = "silver_supplier_basic_AI_csv"


In [1]:
%sql

-- clean up if the tables are already there. 
drop table if exists silver_supplier.supplier_schema.silver_supplier_basic_ai_csv

OK

In [1]:

# check and show the silver table that contains the information about the supplier
cont_df = spark.read.table(f"{silver_catalog}.{silver_schema}.{silver_supplier_basic_Cont_csv}")
cont_df.show()
# create the table that would contain supplier information + the continent
spark.sql("create table if not exists silver_supplier.supplier_schema.silver_supplier_basic_AI_csv (supplier_name_silver  string,  supplier_type_silver  STRING, supplier_country_silver  STRING, supplier_city_silver STRING, continent STRING )")

+--------------------+--------------------+-----------------------+--------------------+---------+
|supplier_name_silver|supplier_type_silver|supplier_country_silver|supplier_city_silver|continent|
+--------------------+--------------------+-----------------------+--------------------+---------+
|24/7 Community Ho...|                NULL|          United States|            Glendale|     NULL|
|     A1 Juice Supply|                NULL|          United States|              Aurora|     NULL|
|            ABC Bank|                NULL|          United States|        Redwood City|     NULL|
|      ABC Consulting|                NULL|              Australia|              Sydney|     NULL|
|              Abbott|                NULL|          United States|         Lake Forest|     NULL|
|       Adams Leasing|                NULL|          United States|           Greenwood|     NULL|
|       Advanced Corp|                NULL|          United States|           BALTIMORE|     NULL|
|        A

+--------------------+--------------------+-----------------------+--------------------+---------+
|supplier_name_silver|supplier_type_silver|supplier_country_silver|supplier_city_silver|continent|
+--------------------+--------------------+-----------------------+--------------------+---------+
|24/7 Community Ho...|                NULL|          United States|            Glendale|     NULL|
|     A1 Juice Supply|                NULL|          United States|              Aurora|     NULL|
|            ABC Bank|                NULL|          United States|        Redwood City|     NULL|
|      ABC Consulting|                NULL|              Australia|              Sydney|     NULL|
|              Abbott|                NULL|          United States|         Lake Forest|     NULL|
|       Adams Leasing|                NULL|          United States|           Greenwood|     NULL|
|       Advanced Corp|                NULL|          United States|           BALTIMORE|     NULL|
|        A

DataFrame[]

In [1]:
%sql

--  Test the Gen AI for its response and modify the prompt as needed.
delete from silver_supplier.supplier_schema.silver_supplier_basic_cont_csv where supplier_name_silver iS NULL;

-- Note that model availablity may change. If this code presents an error and is unable to find the model, pick a different one from the catalog.
SELECT supplier_country_silver, 
       query_model("cohere.command-a-03-2025", 
                   CONCAT("Provide the continent for this country: ", supplier_country_silver)) AS continent 
FROM silver_supplier.supplier_schema.silver_supplier_basic_cont_csv

In [1]:
%sql
-- change the prompt so we get just the continents name. 
delete from silver_supplier.supplier_schema.silver_supplier_basic_cont_csv where supplier_name_silver iS NULL;
select supplier_country_silver , query_model("cohere.command-a-03-2025", concat("Provide just the name of the continent for this country", supplier_country_silver )) as continent from silver_supplier.supplier_schema.silver_supplier_basic_cont_csv

In [1]:
%sql
delete from silver_supplier.supplier_schema.silver_supplier_basic_cont_csv where supplier_name_silver iS NULL;

In [1]:
spark.sql("select * from silver_supplier.supplier_schema.silver_supplier_basic_cont_csv  ").show(50, False)

+-----------------------------------+--------------------+-----------------------+--------------------+---------+
|supplier_name_silver               |supplier_type_silver|supplier_country_silver|supplier_city_silver|continent|
+-----------------------------------+--------------------+-----------------------+--------------------+---------+
|24/7 Community Hospital Group      |NULL                |United States          |Glendale            |NULL     |
|A1 Juice Supply                    |NULL                |United States          |Aurora              |NULL     |
|ABC Bank                           |NULL                |United States          |Redwood City        |NULL     |
|ABC Consulting                     |NULL                |Australia              |Sydney              |NULL     |
|Abbott                             |NULL                |United States          |Lake Forest         |NULL     |
|Adams Leasing                      |NULL                |United States          |Greenw

In [1]:
%sql
-- Read/select from one table and insert into another table with a colmun populated by AI  
-- select * from silver_supplier.supplier_schema.silver_supplier_basic_csv;
-- select supplier_name_silver, supplier_type_silver, supplier_country_silver, supplier_city_silver,  query_model("default.oci_ai_models.cohere.command-a-03-2025",  concat("Just give me the name of the continent of this country:", supplier_country_silver))  as continent from silver_supplier.supplier_schema.silver_supplier_basic_cont_csv ;
insert  into  silver_supplier.supplier_schema.silver_supplier_basic_ai_csv (supplier_name_silver, supplier_type_silver, supplier_country_silver, supplier_city_silver, continent)  select supplier_name_silver, supplier_type_silver, supplier_country_silver, supplier_city_silver,  query_model("default.oci_ai_models.cohere.command-a-03-2025",  concat("Just give me the name of the continent of this country:", supplier_country_silver))  as continent from silver_supplier.supplier_schema.silver_supplier_basic_cont_csv 

OK

In [1]:
%sql
-- print using SQL
select * from silver_supplier.supplier_schema.silver_supplier_basic_ai_csv

In [1]:
## print using spark sql 
spark.sql("select * from silver_supplier.supplier_schema.silver_supplier_basic_ai_csv  ").show(50, False)

+-----------------------------------+--------------------+-----------------------+--------------------+-------------+
|supplier_name_silver               |supplier_type_silver|supplier_country_silver|supplier_city_silver|continent    |
+-----------------------------------+--------------------+-----------------------+--------------------+-------------+
|24/7 Community Hospital Group      |NULL                |United States          |Glendale            |North America|
|A1 Juice Supply                    |NULL                |United States          |Aurora              |North America|
|ABC Bank                           |NULL                |United States          |Redwood City        |North America|
|ABC Consulting                     |NULL                |Australia              |Sydney              |Oceania      |
|Abbott                             |NULL                |United States          |Lake Forest         |North America|
|Adams Leasing                      |NULL               

In [1]:
%sql
-- Add a column named feedback
alter table silver_supplier.supplier_schema.silver_supplier_basic_ai_csv ADD Columns (feedback string)



OK

In [1]:
spark.sql("select * from silver_supplier.supplier_schema.silver_supplier_basic_ai_csv  ").show(10, False)

+-----------------------------------+--------------------+-----------------------+--------------------+-------------+--------+
|supplier_name_silver               |supplier_type_silver|supplier_country_silver|supplier_city_silver|continent    |feedback|
+-----------------------------------+--------------------+-----------------------+--------------------+-------------+--------+
|24/7 Community Hospital Group      |NULL                |United States          |Glendale            |North America|NULL    |
|A1 Juice Supply                    |NULL                |United States          |Aurora              |North America|NULL    |
|ABC Bank                           |NULL                |United States          |Redwood City        |North America|NULL    |
|ABC Consulting                     |NULL                |Australia              |Sydney              |Oceania      |NULL    |
|Abbott                             |NULL                |United States          |Lake Forest         |North Am

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import lit



# Load the table into a DataFrame
df = spark.table("silver_supplier.supplier_schema.silver_supplier_basic_ai_csv")

# Add a new column named "feedback" with a default value of an empty string
df_with_feedback = df.withColumn("feedback", lit(""))

# Show the updated DataFrame
df_with_feedback.show()


+--------------------+--------------------+-----------------------+--------------------+-------------+--------+
|supplier_name_silver|supplier_type_silver|supplier_country_silver|supplier_city_silver|    continent|feedback|
+--------------------+--------------------+-----------------------+--------------------+-------------+--------+
|24/7 Community Ho...|                NULL|          United States|            Glendale|North America|        |
|     A1 Juice Supply|                NULL|          United States|              Aurora|North America|        |
|            ABC Bank|                NULL|          United States|        Redwood City|North America|        |
|      ABC Consulting|                NULL|              Australia|              Sydney|      Oceania|        |
|              Abbott|                NULL|          United States|         Lake Forest|North America|        |
|       Adams Leasing|                NULL|          United States|           Greenwood|North America|  